# Explorer 01 - Ingest and parse

**Purpose.** Read the eight gzipped XML dumps once and save them as normalized Parquet tables. Every other notebook reads the Parquet output and never touches the XML.

**Feeds main report section:** 3. Data and 4. Methodology (pipeline)

**Modules used:** `vinyl.config`, `vinyl.ingest`, `vinyl.tables`

**Tables read:** raw `discogs_<date>_<entity>.xml.gz` files

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

## Why loading is slow

1. **Schema inference.** Without a schema, Spark scans the whole file once to discover the columns, then again to load it. `ingest.read_raw_xml` passes the saved schemas in `schemas/`, which removes the first scan.
2. **Gzip is not splittable.** Each `.xml.gz` file is read by one task on one core, no matter how large the cluster is. The only cure is to do it once: `tables.build_all` stages the in-scope releases as Parquet and builds every child table from that copy.

Expect the releases files to take the longest. Run this notebook once per snapshot and leave it.

In [1]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from vinyl import config, ingest

spark = ingest.get_spark(app_name="vinyl-01-ingest")
from vinyl import tables

## 1. Check paths and schemas

In [2]:
for snap in config.SNAPSHOTS:
    for entity in config.ROW_TAGS:
        print(snap, entity, config.raw_path(snap, entity))

2020 releases /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/raw/discogs_20200101_releases.xml.gz
2020 masters /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/raw/discogs_20200101_masters.xml.gz
2020 labels /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/raw/discogs_20200101_labels.xml.gz
2020 artists /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/raw/discogs_20200101_artists.xml.gz
2026 releases /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/raw/discogs_20261001_releases.xml.gz
2026 masters /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/raw/discogs_20261001_masters.xml.gz
2026 labels /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/raw/discogs_20261001_labels.xml.gz
2026 artists /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/raw/discogs_20261001_artists.xml.gz


In [3]:
# The saved schemas (from the October 2026 files) that replace inference.
for entity in config.ROW_TAGS:
    schema = ingest.load_schema(entity)
    print(f"{entity:9s} {len(schema.fields):2d} top-level fields: {schema.fieldNames()}")

releases  18 top-level fields: ['_id', 'artists', 'companies', 'country', 'data_quality', 'extraartists', 'formats', 'genres', 'identifiers', 'labels', 'master_id', 'notes', 'released', 'series', 'styles', 'title', 'tracklist', 'videos']


masters   10 top-level fields: ['_id', 'artists', 'data_quality', 'genres', 'main_release', 'notes', 'styles', 'title', 'videos', 'year']
labels     8 top-level fields: ['contactinfo', 'data_quality', 'id', 'name', 'parentLabel', 'profile', 'sublabels', 'urls']
artists   10 top-level fields: ['aliases', 'data_quality', 'groups', 'id', 'members', 'name', 'namevariations', 'profile', 'realname', 'urls']


## 2. Quick look at one small raw file

Labels is the smallest file, so it is a cheap test that the reader, row tag, and schema all work before starting the long releases job.

In [4]:
raw_labels = ingest.read_raw_xml(spark, config.MAIN_SNAPSHOT, "labels")
raw_labels.printSchema()
raw_labels.show(5, truncate=60)

root
 |-- contactinfo: string (nullable = true)
 |-- data_quality: string (nullable = true)
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- parentLabel: struct (nullable = true)
 |    |-- _VALUE: string (nullable = true)
 |    |-- _id: long (nullable = true)
 |-- profile: string (nullable = true)
 |-- sublabels: struct (nullable = true)
 |    |-- label: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _VALUE: string (nullable = true)
 |    |    |    |-- _id: long (nullable = true)
 |-- urls: struct (nullable = true)
 |    |-- url: array (nullable = true)
 |    |    |-- element: string (containsNull = true)



+------------------------------------------------------------+------------+---+---------------------+----------------------+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------------------+
|                                                 contactinfo|data_quality| id|                 name|           parentLabel|                                                     profile|                                                   sublabels|                                                        urls|
+------------------------------------------------------------+------------+---+---------------------+----------------------+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------------------+
|Planet E Communications\r\nP.O. Box 27218\r\nDetroit, Mic...|  Needs Vote| 

## 3. Build the Parquet tables

Start with the small tables to confirm the pipeline, then run the release tables. `build_all` reuses `releases_staged` if it already exists, so a failed run does not re-read the XML.

In [5]:
tables.TABLES

{'releases': 'One row per release: title, country, year, master link.',
 'release_genres': 'One row per (release, genre).',
 'release_styles': 'One row per (release, style).',
 'release_formats': 'One row per (release, format) with a reissue flag.',
 'release_labels': 'One row per (release, label) with catalog number.',
 'release_artists': 'One row per (release, main artist).',
 'release_credits': 'One row per (release, credited person, role).',
 'release_companies': 'One row per (release, company), e.g. pressing plants.',
 'labels': 'One row per label with its parent label.',
 'artists': 'One row per artist.',
 'masters': 'One row per master release with its original year.'}

In [6]:
# Small tables first (minutes). fast=True (the default) splits each gz into
# plain-XML chunks so Spark parses them on all cores; the chunks are deleted
# after the Parquet write. 2020 only needs `labels` (tables.snapshot_tables).
for snap in config.SNAPSHOTS:
    wanted = [t for t in tables.snapshot_tables(snap) if t in ("labels", "artists", "masters")]
    print(snap, tables.build_all(spark, snap, tables=wanted, fast=True, keep_stage=False))

2020 {'labels': '/home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/parquet/20200101/labels'}


2026 {'labels': '/home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/parquet/20261001/labels', 'artists': '/home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/parquet/20261001/artists', 'masters': '/home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/parquet/20261001/masters'}


In [7]:
# Release tables (the long step). Run once per snapshot. fast=True splits the
# gz, drops releases dated outside 2016-2026 before parsing, and skips the
# unused fields in config.RELEASE_DROP_FIELDS. 2020 only builds the tables the
# snapshot comparison needs (tables.snapshot_tables).
for snap in config.SNAPSHOTS:
    wanted = [t for t in tables.snapshot_tables(snap) if t.startswith("release")]
    written = tables.build_all(spark, snap, tables=wanted, fast=True, keep_stage=False)
    for name, path in written.items():
        print(snap, name, path)

2020 releases /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/parquet/20200101/releases
2020 release_labels /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/parquet/20200101/release_labels


2026 releases /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/parquet/20261001/releases
2026 release_genres /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/parquet/20261001/release_genres
2026 release_styles /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/parquet/20261001/release_styles
2026 release_formats /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/parquet/20261001/release_formats
2026 release_labels /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/parquet/20261001/release_labels
2026 release_artists /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/parquet/20261001/release_artists
2026 release_credits /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/parquet/20261001/release_credits
2026 release_companies /home2/bsdsba2028/jsandiego/bdcc-lab2/lab2/data/parquet/20261001/release_companies


## 4. Sanity checks on the output

In [8]:
# Row counts per table and snapshot. Tables a snapshot does not build
# (tables.snapshot_tables) show as NaN.
import pandas as pd

rows = []
for snap in config.SNAPSHOTS:
    for name in tables.snapshot_tables(snap):
        rows.append({"snapshot": snap, "table": name, "rows": ingest.read_table(spark, snap, name).count()})
pd.DataFrame(rows).pivot(index="table", columns="snapshot", values="rows").reindex(list(tables.TABLES))

snapshot,2020,2026
table,,
releases,1082351.0,3829109.0
release_genres,NaN,4790607.0
release_styles,NaN,6488970.0
release_formats,NaN,3985674.0
release_labels,1208158.0,4262009.0
release_artists,NaN,4485710.0
release_credits,NaN,15238351.0
release_companies,NaN,4598779.0
labels,1461523.0,2426075.0


In [9]:
# Releases per year in the main snapshot: are all years 2016-2026 present?
releases = ingest.read_table(spark, config.MAIN_SNAPSHOT, "releases")
releases.groupBy("year").count().orderBy("year").show()

+----+------+
|year| count|
+----+------+
|2016|385281|
|2017|390355|
|2018|396710|
|2019|394016|
|2020|415671|
|2021|391297|
|2022|360863|
|2023|347065|
|2024|312179|
|2025|280882|
|2026|154790|
+----+------+



In [10]:
# TODO: how many raw releases were dropped for having no usable `released` date?
# TODO: null rates of release_id, year, master_id, country.
# TODO: confirm the 2020 files parse with the 2026 schemas (one inferred read on a sample).

## Decisions to carry into `main.ipynb`

Filled in from the full-data runs in `executed/`. Each line should end up as a sentence in the report's methodology or limitations.

- [x] Number of releases in scope per snapshot: 3,829,109 in October 2026 and 1,082,351 in January 2020, all dated 2016-2026 (row counts in section 4).
- [ ] How many raw releases were dropped for missing dates - not decided: the year pre-filter runs while the XML is split, and its statistics were not kept for the full runs, so the count is not available (the `year_filter` row in main.ipynb 3.4 says so).
- [x] Whether the January 2020 files match the saved schemas: yes in practice; all 2020 tables (releases, release_labels, labels, artists, masters) were built with the saved October 2026 schemas without parse errors. The separate inferred-schema check on a sample (the third comment in the last code cell of section 4) was not run.
- [x] Spark version, cluster size, and wall-clock time: Spark 4.1.1 in local mode on one JupyterHub pod (4 CPUs, 16 GiB memory limit); the one-time conversion of both snapshots took 3 h 17 min (T03 run, `executed/README.md`).